In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import  transforms, datasets
from torch.utils.data import Dataset, DataLoader, random_split
import matplotlib.pyplot as plt
from PIL import Image
import pywt

import time
import random
from best_model1 import Vision

def benchmark_inference_speed(model, dataset, device='cpu'):
    model.eval()
    model.to(device)

    batch_sizes = [1, 2, 4, 8, 16, 32, 64, 128]
    warmup_iterations = 5
    benchmark_iterations = 10

    print(f"Начинаем замер производительности на устройстве: {device}")
    print("-" * 65)
    print(f"{'Размер батча':<15} | {'Среднее время/батч (мс)':<25} | {'Картинок/сек':<20}")
    print("-" * 65)

    results = {}

    for batch_size in batch_sizes:
        if len(dataset) < batch_size:
            print(f"Пропуск батча {batch_size}, т.к. датасет ({len(dataset)}) слишком мал.")
            continue

        indices = random.sample(range(len(dataset)), batch_size)
        batch_list = [dataset[i][0] for i in indices]
        input_batch = torch.stack(batch_list).to(device)

        with torch.no_grad():
            for _ in range(warmup_iterations):
                _ = model(input_batch)

        if device in ['cuda', 'mps']:
            torch.mps.synchronize() if device == 'mps' else torch.cuda.synchronize()

        start_time = time.perf_counter()
        with torch.no_grad():
            for _ in range(benchmark_iterations):
                _ = model(input_batch)
        
        if device in ['cuda', 'mps']:
            torch.mps.synchronize() if device == 'mps' else torch.cuda.synchronize()
            
        end_time = time.perf_counter()

        total_time = end_time - start_time
        avg_time_per_batch_ms = (total_time / benchmark_iterations) * 1000
        images_per_second = (batch_size * benchmark_iterations) / total_time

        results[batch_size] = {
            'avg_time_ms': avg_time_per_batch_ms,
            'throughput_ips': images_per_second
        }
        
        print(f"{batch_size:<15} | {avg_time_per_batch_ms:<25.2f} | {images_per_second:<20.2f}")

    print("-" * 65)
    return results

In [ ]:
class ImageRestorationDataset(Dataset):
    def __init__(self, degraded_dir, clean_dir, transform=None):
        self.degraded_dir = degraded_dir
        self.clean_dir = clean_dir
        self.transform = transform
        
        self.image_files = sorted(os.listdir(degraded_dir))

    def __len__(self):
        return 10000

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        
        degraded_img_path = os.path.join(self.degraded_dir, img_name)
        clean_img_path = os.path.join(self.clean_dir, img_name)
        
        degraded_image = Image.open(degraded_img_path).convert('RGB')
        clean_image = Image.open(clean_img_path).convert('RGB')
        
        if self.transform:
            degraded_image = self.transform(degraded_image)
            clean_image = self.transform(clean_image)
            
        return degraded_image, clean_image

DATASET_PATH = 'dataset_patches' 
DEGRADED_DIR = os.path.join(DATASET_PATH, 'degraded')
CLEAN_DIR = os.path.join(DATASET_PATH, 'clean')

transform = transforms.Compose([
    transforms.ToTensor()
])

full_dataset = ImageRestorationDataset(
    degraded_dir=DEGRADED_DIR,
    clean_dir=CLEAN_DIR,
    transform=transform
)

train_size = int(0.9 * len(full_dataset))
val_size = len(full_dataset) - train_size
data_train, data_val = random_split(full_dataset, [train_size, val_size])

print(f"Размер обучающей выборки: {len(data_train)}")
print(f"Размер валидационной выборки: {len(data_val)}")

In [ ]:
benchmark_inference_speed(Vision, data_val, 'mps')